## 라이브러리 호출

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 데이터 로드

In [27]:
# 원본 데이터 로드
df_platinum_Match = pd.read_csv('../../tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('../../tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('../../tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('../../tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('../../tft_game_dataset/TFT_Challenger_MatchData.csv')

df_Champion_info = pd.read_csv('../../tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_Items_info = pd.read_csv('../../tft_game_dataset/TFT_Item_CurrentVersion.csv')


# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

---
## Match 관련 데이터 병합 (행 기준)

In [29]:
# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name

# 티어가 잘 추가되었는지 확인하는 용도
for name, df in match_data.items():
    print(f"=== {name} ===")
    print(df['tier'].value_counts())
    print()

=== platinum ===
tier
platinum    80000
Name: count, dtype: int64

=== diamond ===
tier
diamond    80000
Name: count, dtype: int64

=== master ===
tier
master    79999
Name: count, dtype: int64

=== grand_master ===
tier
grand_master    80000
Name: count, dtype: int64

=== challenger ===
tier
challenger    79999
Name: count, dtype: int64



In [30]:
# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(2)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum


In [31]:
len(df_all_match[df_all_match['Ranked']==0]['gameId'])

92

In [32]:
print(f"'ranked==0' 행 수: {len(df_all_match[df_all_match['Ranked']==0])}")
print(f"전체 행 수: {len(df_all_match)}")
print(f"비율: {round(len(df_all_match[df_all_match['Ranked']==0]) / len(df_all_match) * 100, 3)}%")

'ranked==0' 행 수: 92
전체 행 수: 399998
비율: 0.023%


In [33]:
df_all_match[df_all_match['Ranked']==0]['gameId'].unique()

<StringArray>
['KR_4229542485', 'KR_4327058667', 'KR_4327053826', 'KR_4236285365',
 'KR_3890408252', 'KR_4231144753', 'KR_4323594767', 'KR_3891808329',
 'KR_3891442705', 'KR_3891371111', 'KR_4231208133', 'KR_4295022760',
 'KR_4303195386', 'KR_4336328756']
Length: 14, dtype: str

In [34]:
# 게임 아이디별로 ranked == 0이라서 지워지는 행 수
# size() : 그룹별 행 수
df_all_match[df_all_match['Ranked']==0].groupby('gameId').size()

gameId
KR_3890408252    8
KR_3891371111    8
KR_3891442705    8
KR_3891808329    8
KR_4229542485    7
KR_4231144753    7
KR_4231208133    7
KR_4236285365    7
KR_4295022760    7
KR_4303195386    7
KR_4323594767    7
KR_4327053826    2
KR_4327058667    2
KR_4336328756    7
dtype: int64

In [46]:
len(df_all_match[df_all_match['gameId'].isin(['KR_4336328756'])]) # 1,2 데이터 손실

8

In [24]:
df_all_match[df_all_match['Ranked']==0].groupby(['gameId', 'tier']).size()

gameId         tier       
KR_3890408252  Platinum       8
KR_3891371111  Platinum       8
KR_3891442705  Platinum       8
KR_3891808329  Platinum       8
KR_4229542485  Platinum       7
KR_4231144753  Platinum       7
KR_4231208133  Platinum       7
KR_4236285365  Platinum       7
KR_4295022760  Master         7
KR_4303195386  Master         7
KR_4323594767  Platinum       7
KR_4327053826  Platinum       2
KR_4327058667  Platinum       2
KR_4336328756  GrandMaster    7
dtype: int64